In [1]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from datetime import datetime, timedelta
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')

In [3]:
#set style
plt.style.use('seaborn-v0_8')
sns.set_palette('viridis')

#connect to database
conn = sqlite3.connect('../data/database/financial_data.db')

#load data
clients = pd.read_sql_query('SELECT * FROM clients', conn)
trades = pd.read_sql_query('SELECT * FROM trades', conn)
accounts = pd.read_sql_query('SELECT * FROM accounts', conn)
transactions = pd.read_sql_query('SELECT * FROM transactions', conn)

In [4]:
# --- preview data ---
print("\n🔎Dataset Previews:")
print("Clients:")
display(clients.head())
print("Trades:")
display(trades.head())
print("Accounts:")
display(accounts.head())
print("Transactions:")
display(transactions.head())


🔎Dataset Previews:
Clients:


,client_id,age,income_bracket,risk_tolerance,investment_experience,join_date,country
0,C10000,54,Medium,Very High,Beginner,2020-02-16 00:00:00,Canada
1,C10001,61,Medium,Low,Beginner,2025-11-15 00:00:00,Canada
2,C10002,27,Medium,Very High,Intermediate,2024-08-17 00:00:00,Germany
3,C10003,23,Low,Very High,Intermediate,2022-02-03 00:00:00,Germany
4,C10004,57,Medium,High,Advanced,2025-11-07 00:00:00,Australia


Trades:


,transaction_id,client_id,symbol,transaction_date,type,quantity,price,amount
0,T100000,C10298,MSFT,2025-05-04 00:00:00,Buy,192,278.68,53505.91
1,T100001,C10720,AMZN,2024-10-19 00:00:00,Sell,37,132.04,4885.52
2,T100002,C10720,NVDA,2025-08-21 00:00:00,Sell,60,335.72,20143.46
3,T100003,C10968,AMZN,2025-12-11 00:00:00,Sell,167,107.63,17974.13
4,T100004,C10254,TSLA,2020-11-29 00:00:00,Buy,88,172.50,15180.05


Accounts:


,client_id,account_balance,last_updated
0,C10000,11119,2025-09-16 18:10:22.208536
1,C10001,27182,2025-09-16 18:10:22.214717
2,C10002,46465,2025-09-16 18:10:22.216712
3,C10003,25315,2025-09-16 18:10:22.217713
4,C10004,21668,2025-09-16 18:10:22.218707


Transactions:


,client_id,date,type,amount
0,C10000,2020-02-16 00:00:00,Deposit,11119.0
1,C10000,2021-08-12 00:00:00,Withdrawal,2118.0
2,C10000,2024-04-10 00:00:00,Withdrawal,3869.0
3,C10000,2021-06-03 00:00:00,Deposit,175.0
4,C10000,2024-12-19 00:00:00,Deposit,5174.0


In [5]:
#data cleaning and validation
print("\n📊 Initial data shapes:")
print(f"Clients: {clients.shape}")
print(f"Trades: {trades.shape}")
print(f"Accounts: {accounts.shape}")
print(f"Transactions: {transactions.shape}")


📊 Initial data shapes:
Clients: (1000, 7)
Trades: (100000, 8)
Accounts: (1000, 3)
Transactions: (5515, 4)


In [6]:
#check for missing values
print("\n🛑 Missing values:")
print("Clients:")
display(clients.isnull().sum())
print("Trades:")
display(trades.isnull().sum())
print("Accounts:")
display(accounts.isnull().sum())
print("Transactions:")
display(transactions.isnull().sum())



🛑 Missing values:
Clients:


client_id                0
age                      0
income_bracket           0
risk_tolerance           0
investment_experience    0
join_date                0
country                  0
dtype: int64

Trades:


transaction_id      0
client_id           0
symbol              0
transaction_date    0
type                0
quantity            0
price               0
amount              0
dtype: int64

Accounts:


client_id          0
account_balance    0
last_updated       0
dtype: int64

Transactions:


client_id    0
date         0
type         0
amount       0
dtype: int64

In [7]:
#data type conversion
clients['join_date'] = pd.to_datetime(clients['join_date'], errors='coerce')
trades['transaction_date'] = pd.to_datetime(trades['transaction_date'], errors='coerce')
transactions['date'] = pd.to_datetime(transactions['date'], errors='coerce')
accounts['last_updated'] = pd.to_datetime(accounts['last_updated'], errors='coerce')

#create additional features
#client tenure
current_date = pd.Timestamp.now()
clients['tenure_days'] = (pd.Series(current_date, index=clients.index) - clients['join_date']).dt.days

#trade month and week
trades['transaction_month'] = trades['transaction_date'].dt.to_period('M')
trades['transaction_week']  = trades['transaction_date'].dt.isocalendar().week  # type: ignore

#transaction month
transactions['transaction_month'] = transactions['date'].dt.to_period('M')


In [8]:
#calculate client trading metrics
client_trade_metrics = trades.groupby('client_id').agg({
    'transaction_id': 'count',
    'amount': ['sum', 'mean', 'std'],
    'quantity': ['sum', 'mean']
}).round(2)

client_trade_metrics.columns = [
    'trade_count', 'total_trade_value', 'avg_trade_value',
    'std_trade_value', 'total_quantity', 'avg_quantity'
]
client_trade_metrics = client_trade_metrics.reset_index()

print("\n📊 Client Trade Metrics Preview:")
display(client_trade_metrics.head())


📊 Client Trade Metrics Preview:


,client_id,trade_count,total_trade_value,avg_trade_value,std_trade_value,total_quantity,avg_quantity
0,C10000,101,2070146.74,20496.50,14622.27,10030,99.31
1,C10001,98,2466664.64,25170.05,18070.90,10828,110.49
2,C10002,91,2056016.16,22593.58,16463.27,9686,106.44
3,C10003,92,1998764.47,21725.70,17154.08,9172,99.70
4,C10004,95,1841425.29,19383.42,15530.80,8619,90.73


In [9]:
#calculate client transaction metrics
client_transaction_metrics = transactions.groupby('client_id').agg({
    'amount': ['sum', 'count'],
    'type': lambda x: (x == 'Deposit').sum() / len(x) #deposit ratio
}).round(2)

client_transaction_metrics.columns = [
    'total_transaction_value', 'transaction_count', 'deposit_ratio'
]
client_transaction_metrics = client_transaction_metrics.reset_index()

print("\n📊 Client Transaction Metrics Preview:")
display(client_transaction_metrics.head())


📊 Client Transaction Metrics Preview:


,client_id,total_transaction_value,transaction_count,deposit_ratio
0,C10000,52091.0,10,0.80
1,C10001,27182.0,1,1.00
2,C10002,46815.0,2,0.50
3,C10003,34596.0,3,0.67
4,C10004,21668.0,1,1.00


In [10]:
#merge all client data
client_merged = clients.merge(accounts, on='client_id', how='left')
client_merged = client_merged.merge(client_trade_metrics, on='client_id', how='left')
client_merged = client_merged.merge(client_transaction_metrics, on='client_id', how='left')

#fill NaN values
client_merged.fillna({
    'trade_count': 0,
    'total_trade_value': 0,
    'avg_trade_value': 0,
    'std_trade_value': 0,
    'total_quantity': 0,
    'avg_quantity': 0,
    'total_transaction_value': 0,
    'transaction_count': 0,
    'deposit_ratio': 0
}, inplace=True)

#calculate additional metrics
client_merged['avg_daily_trades'] = client_merged['trade_count'] / client_merged['tenure_days'].clip(lower=1)
client_merged['value_per_trade'] = client_merged['total_trade_value'] / client_merged['trade_count'].replace(0, 1)

print("\n✅ Final Client Dataset Preview:")
display(client_merged.head())


✅ Final Client Dataset Preview:


,client_id,age,income_bracket,risk_tolerance,investment_experience,join_date,country,tenure_days,account_balance,last_updated,...,total_trade_value,avg_trade_value,std_trade_value,total_quantity,avg_quantity,total_transaction_value,transaction_count,deposit_ratio,avg_daily_trades,value_per_trade
0,C10000,54,Medium,Very High,Beginner,2020-02-16,Canada,2040,11119,2025-09-16 18:10:22.208536,...,2070146.74,20496.50,14622.27,10030,99.31,52091.0,10,0.80,0.049510,20496.502376
1,C10001,61,Medium,Low,Beginner,2025-11-15,Canada,-59,27182,2025-09-16 18:10:22.214717,...,2466664.64,25170.05,18070.90,10828,110.49,27182.0,1,1.00,98.000000,25170.047347
2,C10002,27,Medium,Very High,Intermediate,2024-08-17,Germany,396,46465,2025-09-16 18:10:22.216712,...,2056016.16,22593.58,16463.27,9686,106.44,46815.0,2,0.50,0.229798,22593.584176
3,C10003,23,Low,Very High,Intermediate,2022-02-03,Germany,1322,25315,2025-09-16 18:10:22.217713,...,1998764.47,21725.70,17154.08,9172,99.70,34596.0,3,0.67,0.069592,21725.700761
4,C10004,57,Medium,High,Advanced,2025-11-07,Australia,-51,21668,2025-09-16 18:10:22.218707,...,1841425.29,19383.42,15530.80,8619,90.73,21668.0,1,1.00,95.000000,19383.424105


In [11]:
#save processed data
client_merged.to_csv('../data/processed/client_analysis_data.csv', index=False)
trades.to_csv('../data/processed/trades_processed.csv', index=False)
transactions.to_csv('../data/processed/transactions_processed.csv', index=False)

print("\n🎉 Data cleaning and processing complete!")
print(f"Final client dataset shape: {client_merged.shape}")


🎉 Data cleaning and processing complete!
Final client dataset shape: (1000, 21)
